In [1]:
# Import libraries
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# Check environment and require GPU
print("python:", sys.executable)
print("torch:", torch.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook. Please run with the weather-predict env and a CUDA PyTorch build.")

device = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0))

python: d:\anaconda\envs\weather-predict\python.exe
torch: 2.11.0+cu128
device: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
# Read train, valid, test data
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(processed_dir / "train.csv")
valid_df = pd.read_csv(processed_dir / "valid.csv")
test_df = pd.read_csv(processed_dir / "test.csv")
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

train_df.head()

,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,hour,month,...,cloud_cover_lag_2h,cloud_cover_lag_3h,temperature_2m_lag_1h,temperature_2m_lag_2h,temperature_2m_lag_3h,precipitation_rolling_3h,precipitation_rolling_6h,rain_next_1h,rain_next_2h,rain_next_3h
0,2019-01-01 05:00:00,0.0,64,1026.0,8.8,15.0,17,100,5,1,...,34.0,39.0,9.0,9.1,9.7,0.0,0.0,0,0,0
1,2019-01-01 06:00:00,0.0,64,1026.4,8.9,14.6,16,100,6,1,...,59.0,34.0,8.8,9.0,9.1,0.0,0.0,0,0,0
2,2019-01-01 07:00:00,0.0,71,1026.5,9.2,15.8,24,100,7,1,...,100.0,59.0,8.9,8.8,9.0,0.0,0.0,0,0,0
3,2019-01-01 08:00:00,0.0,67,1027.4,9.8,15.6,23,100,8,1,...,100.0,100.0,9.2,8.9,8.8,0.0,0.0,0,0,0
4,2019-01-01 09:00:00,0.0,65,1028.3,10.3,14.8,18,100,9,1,...,100.0,100.0,9.8,9.2,8.9,0.0,0.0,0,0,0


In [4]:
# Add sin cos features and choose columns
target_cols = ["rain_next_1h", "rain_next_2h", "rain_next_3h"]

def add_cyclic_features(df):
    df = df.copy()
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * (df["month"] - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (df["month"] - 1) / 12)
    df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["wind_dir_sin"] = np.sin(2 * np.pi * df["wind_direction_10m"] / 360)
    df["wind_dir_cos"] = np.cos(2 * np.pi * df["wind_direction_10m"] / 360)
    return df

train_df = add_cyclic_features(train_df)
valid_df = add_cyclic_features(valid_df)
test_df = add_cyclic_features(test_df)
train_valid_df = add_cyclic_features(train_valid_df)

feature_cols = [
    "precipitation",
    "relative_humidity_2m",
    "surface_pressure",
    "temperature_2m",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation_rolling_3h",
    "precipitation_rolling_6h",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "dayofweek_sin",
    "dayofweek_cos",
    "wind_dir_sin",
    "wind_dir_cos",
]

print("num_features:", len(feature_cols))
feature_cols

num_features: 16


['precipitation',
 'relative_humidity_2m',
 'surface_pressure',
 'temperature_2m',
 'wind_speed_10m',
 'cloud_cover',
 'precipitation_rolling_3h',
 'precipitation_rolling_6h',
 'hour_sin',
 'hour_cos',
 'month_sin',
 'month_cos',
 'dayofweek_sin',
 'dayofweek_cos',
 'wind_dir_sin',
 'wind_dir_cos']

In [5]:
# Scale features and create 24-hour sequences
seq_len = 24
batch_size = 256

def scale_data(fit_df, data_df, scaler=None):
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(fit_df[feature_cols])

    scaled_df = data_df.copy()
    scaled_df[feature_cols] = scaler.transform(data_df[feature_cols]).astype(np.float32)
    return scaled_df, scaler

def make_sequences(df):
    x_values = df[feature_cols].values.astype(np.float32)
    y_values = df[target_cols].values.astype(np.float32)
    times = df["datetime"].values
    x_seq = []
    y_seq = []
    t_seq = []

    for i in range(seq_len - 1, len(df)):
        x_seq.append(x_values[i - seq_len + 1:i + 1])
        y_seq.append(y_values[i])
        t_seq.append(times[i])

    return np.array(x_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32), np.array(t_seq)

def make_loader(x, y, shuffle=False):
    dataset = TensorDataset(torch.tensor(x), torch.tensor(y))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, pin_memory=True)

train_scaled, scaler = scale_data(train_df, train_df)
valid_scaled, _ = scale_data(train_df, valid_df, scaler)

x_train, y_train, _ = make_sequences(train_scaled)
x_valid, y_valid, _ = make_sequences(valid_scaled)

train_loader = make_loader(x_train, y_train, shuffle=True)
valid_loader = make_loader(x_valid, y_valid)

x_train.shape, y_train.shape, x_valid.shape, y_valid.shape

((43796, 24, 16), (43796, 3), (8761, 24, 16), (8761, 3))

In [6]:
# Define LSTM model and train functions
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_size=64, output_dim=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1])

def loss_on_loader(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    total_rows = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            loss = loss_fn(model(xb), yb)
            total_loss += loss.item() * len(xb)
            total_rows += len(xb)

    return total_loss / total_rows

def train_model(train_loader, input_dim, epochs=30, lr=0.001, valid_loader=None):
    model = LSTMModel(input_dim).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_loss = float("inf")
    best_state = None
    best_epoch = epochs

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        total_rows = 0

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)
            total_rows += len(xb)

        train_loss = total_loss / total_rows
        valid_loss = np.nan

        if valid_loader is not None:
            valid_loss = loss_on_loader(model, valid_loader, loss_fn)
            if valid_loss < best_loss:
                best_loss = valid_loss
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_epoch

In [7]:
# Train on train and choose best epoch by valid loss
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

epochs = 30
lr = 0.001
input_dim = len(feature_cols)

valid_model, history_df, best_epoch = train_model(train_loader, input_dim, epochs=epochs, lr=lr, valid_loader=valid_loader)
history_df.to_csv(results_dir / "lstm_training_history.csv", index=False)

print("best_epoch:", best_epoch)
history_df.tail()

best_epoch: 7


,epoch,train_loss,valid_loss
25,26,0.208874,0.476622
26,27,0.205453,0.470505
27,28,0.200461,0.492778
28,29,0.195652,0.491467
29,30,0.192451,0.506746


In [8]:
# Prediction and metric functions
def predict_proba(model, x, batch_size=512):
    model.eval()
    probs = []

    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            xb = torch.tensor(x[start:start + batch_size], dtype=torch.float32).to(device)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())

    return np.vstack(probs)

def evaluate_targets(y_true, y_prob, y_pred, split_name):
    rows = []

    for i, target in enumerate(target_cols):
        tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred[:, i], labels=[0, 1]).ravel()
        rows.append({
            "model": "lstm",
            "split": split_name,
            "target": target,
            "threshold": 0.5,
            "log_loss": log_loss(y_true[:, i], y_prob[:, i], labels=[0, 1]),
            "brier_score": brier_score_loss(y_true[:, i], y_prob[:, i]),
            "roc_auc": roc_auc_score(y_true[:, i], y_prob[:, i]),
            "pr_auc": average_precision_score(y_true[:, i], y_prob[:, i]),
            "accuracy": accuracy_score(y_true[:, i], y_pred[:, i]),
            "precision": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "recall": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "f1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        })

    return pd.DataFrame(rows)

In [9]:
# Train final model on train valid and test
train_valid_scaled, final_scaler = scale_data(train_valid_df, train_valid_df)
test_scaled, _ = scale_data(train_valid_df, test_df, final_scaler)

x_train_valid, y_train_valid, _ = make_sequences(train_valid_scaled)
x_test, y_test, test_times = make_sequences(test_scaled)

train_valid_loader = make_loader(x_train_valid, y_train_valid, shuffle=True)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
final_model, _, _ = train_model(train_valid_loader, input_dim, epochs=best_epoch, lr=lr)

test_prob = predict_proba(final_model, x_test)
test_pred = (test_prob >= 0.5).astype(int)
metrics_df = evaluate_targets(y_test, test_prob, test_pred, "test")

test_predictions = pd.DataFrame({"datetime": test_times})
for i, target in enumerate(target_cols):
    test_predictions[f"{target}_true"] = y_test[:, i].astype(int)
    test_predictions[f"{target}_prob"] = test_prob[:, i]
    test_predictions[f"{target}_pred"] = test_pred[:, i]

metrics_df.to_csv(results_dir / "lstm_test_metrics.csv", index=False)
test_predictions.to_csv(results_dir / "lstm_test_predictions.csv", index=False)

metrics_df

,model,split,target,threshold,log_loss,brier_score,roc_auc,pr_auc,accuracy,precision,recall,f1,tn,fp,fn,tp
0,lstm,test,rain_next_1h,0.5,0.324813,0.101701,0.912174,0.811469,0.856194,0.782677,0.639091,0.703634,5987,414,842,1491
1,lstm,test,rain_next_2h,0.5,0.372996,0.119417,0.883165,0.739378,0.827685,0.727473,0.567510,0.637611,5905,496,1009,1324
2,lstm,test,rain_next_3h,0.5,0.392957,0.127065,0.869352,0.701441,0.814174,0.699663,0.533219,0.605206,5867,534,1089,1244
